# Donut fine-tuning on CORD v2

End-to-end notebook: loads `naver-clova-ix/donut-base`, fine-tunes on `naver-clova-ix/cord-v2`, and saves the trained model.

Adds light training-time image augmentation (brightness / contrast / tiny rotation) so the same images contribute more signal across epochs.

**Designed for Lightning AI Studio** (single GPU, ~24 GB VRAM e.g. L4/A10). Adjust `IMAGE_SIZE` or `EPOCHS` in the config cell if you have a smaller/larger GPU.

Expected runtime: ~45 min on L4 / ~25 min on A100 for 5 epochs.

## 1. Setup

In [1]:
%pip install -q "transformers==4.44.2" "datasets==2.20.0" "accelerate==0.34.2" sentencepiece pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import random
import re
from typing import Any, Callable

import torch
from torch.utils.data import Dataset
from PIL import Image, ImageEnhance
from datasets import load_dataset
from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("device: CPU (training will be very slow)")

torch: 2.12.0+cu130
device: NVIDIA L40S
bf16 supported: True


In [3]:
# Each spec: how to load one dataset and how to extract a gt_parse dict from
# its ground_truth column. Add more entries to mix in additional datasets.
DATASET_SPECS = [
    {
        "name": "naver-clova-ix/cord-v2",
        "train_split": "train",
        "val_split": "validation",
        "image_col": "image",
        "gt_col": "ground_truth",
        "gt_parser": lambda gt: json.loads(gt)["gt_parse"],
        "required": True,
    },
]

class Cfg:
    BASE_MODEL = "naver-clova-ix/donut-base"
    TASK_TOKEN = "<s_cord-v2>"
    IMAGE_SIZE = [1280, 960]  # [height, width]
    MAX_LENGTH = 768           # decoder cap
    OUTPUT_DIR = "./donut-cord-finetuned"
    EPOCHS = 5
    LR = 3e-5
    PER_DEVICE_BATCH = 1
    GRAD_ACC = 4               # effective batch = 4
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    SEED = 42
    USE_AUGMENT = True         # light brightness/contrast/rotation aug

set_seed(Cfg.SEED)
os.makedirs(Cfg.OUTPUT_DIR, exist_ok=True)
print("output dir:", Cfg.OUTPUT_DIR)

output dir: ./donut-cord-finetuned


## 2. Load datasets

Each dataset is loaded independently. Optional ones (`required=False`) that fail to download are skipped with a warning so the notebook keeps running with whatever it has.

In [4]:
loaded_specs = []  # list of (spec, train_split, val_split)

for spec in DATASET_SPECS:
    try:
        ds = load_dataset(spec["name"])
        train_split = ds[spec["train_split"]]
        val_split = ds[spec["val_split"]] if spec["val_split"] in ds else None
        loaded_specs.append((spec, train_split, val_split))
        print(
            f"loaded {spec['name']}: train={len(train_split)}"
            f" val={len(val_split) if val_split else 0}"
        )
    except Exception as exc:
        if spec["required"]:
            raise
        print(f"skipping optional dataset {spec['name']}: {exc}")

assert loaded_specs, "no datasets loaded"
total_train = sum(len(t) for _, t, _ in loaded_specs)
total_val = sum(len(v) for _, _, v in loaded_specs if v is not None)
print(f"\ncombined train: {total_train}  val: {total_val}")

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

loaded naver-clova-ix/cord-v2: train=800 val=100

combined train: 800  val: 100


In [5]:
# Peek at one example per loaded dataset to confirm the parser works.
for spec, train_split, _ in loaded_specs:
    print(f"\n=== {spec['name']} ===")
    sample = train_split[0]
    gt = spec["gt_parser"](sample[spec["gt_col"]])
    print(json.dumps(gt, indent=2, ensure_ascii=False)[:400])
    print("image size:", sample[spec["image_col"]].size)


=== naver-clova-ix/cord-v2 ===
{
  "menu": [
    {
      "nm": "Nasi Campur Bali",
      "cnt": "1 x",
      "price": "75,000"
    },
    {
      "nm": "Bbk Bengil Nasi",
      "cnt": "1 x",
      "price": "125,000"
    },
    {
      "nm": "MilkShake Starwb",
      "cnt": "1 x",
      "price": "37,000"
    },
    {
      "nm": "Ice Lemon Tea",
      "cnt": "1 x",
      "price": "24,000"
    },
    {
      "nm": "Nasi Ayam Dewa
image size: (864, 1296)


## 3. Model + processor

We resize the Donut encoder to a smaller image (1280×960) to fit a single 24 GB GPU and cap the decoder at 1024 tokens (room for invoice-sized parses).

In [8]:
# use_fast=False forces the slow (sentencepiece) tokenizer and avoids the
# slow->fast conversion that needs the protobuf library.
processor = DonutProcessor.from_pretrained(Cfg.BASE_MODEL, use_fast=False)
model = VisionEncoderDecoderModel.from_pretrained(Cfg.BASE_MODEL)

processor.image_processor.size = {"height": Cfg.IMAGE_SIZE[0], "width": Cfg.IMAGE_SIZE[1]}
processor.image_processor.do_align_long_axis = False
model.config.encoder.image_size = Cfg.IMAGE_SIZE
model.config.decoder.max_length = Cfg.MAX_LENGTH

print("image size:", processor.image_processor.size)
print("decoder max_length:", model.config.decoder.max_length)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/809M [00:00<?, ?B/s]

image size: {'height': 1280, 'width': 960}
decoder max_length: 768


## 4. JSON ↔ token conversion

Donut consumes a target sequence built from XML-like tags wrapping the structured fields:
```
<s_receipt><s_menu><s_nm>Latte</s_nm><s_price>9000</s_price></s_menu>...</s>
```
We walk every loaded training split once to collect every `<s_KEY>` / `</s_KEY>` pair, add them to the tokenizer, and resize the decoder embeddings — so the vocab covers fields from all datasets, not just CORD.

In [9]:
def json2token(obj: Any, sort_keys: bool = True) -> str:
    if isinstance(obj, dict):
        if len(obj) == 1 and "text_sequence" in obj:
            return obj["text_sequence"]
        keys = sorted(obj.keys()) if sort_keys else list(obj.keys())
        out = ""
        for k in keys:
            out += f"<s_{k}>" + json2token(obj[k], sort_keys) + f"</s_{k}>"
        return out
    if isinstance(obj, list):
        return "<sep/>".join(json2token(x, sort_keys) for x in obj)
    return str(obj)


def collect_keys(obj: Any, out: set) -> None:
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.add(k)
            collect_keys(v, out)
    elif isinstance(obj, list):
        for x in obj:
            collect_keys(x, out)


def safe_parse(spec, gt_value):
    """Run a spec's gt_parser, returning None on failure so the example is skipped."""
    try:
        parsed = spec["gt_parser"](gt_value)
        if isinstance(parsed, dict):
            return parsed
        return None
    except Exception:
        return None

In [10]:
all_keys: set = set()
kept_per_dataset = {}
for spec, train_split, _ in loaded_specs:
    kept = 0
    for gt_value in train_split[spec["gt_col"]]:
        parsed = safe_parse(spec, gt_value)
        if parsed is not None:
            collect_keys(parsed, all_keys)
            kept += 1
    kept_per_dataset[spec["name"]] = kept
    print(f"{spec['name']}: parsed {kept}/{len(train_split)} train examples")

print(f"\ndiscovered {len(all_keys)} unique keys across all datasets")
print("sample keys:", sorted(all_keys)[:20])

# Special tokens (task start + JSON list separator)
processor.tokenizer.add_special_tokens({"additional_special_tokens": [Cfg.TASK_TOKEN]})

# Field-tag tokens — sorted for reproducibility
field_tokens = []
for k in sorted(all_keys):
    field_tokens.append(f"<s_{k}>")
    field_tokens.append(f"</s_{k}>")
field_tokens.append("<sep/>")
added = processor.tokenizer.add_tokens(field_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Wire the task-start and pad tokens into the model config so generation works.
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(Cfg.TASK_TOKEN)
model.config.eos_token_id = processor.tokenizer.eos_token_id

print(f"\nvocab size after add: {len(processor.tokenizer)} (+{added} field tokens)")
print("decoder_start_token_id:", model.config.decoder_start_token_id)

naver-clova-ix/cord-v2: parsed 800/800 train examples

discovered 27 unique keys across all datasets
sample keys: ['cashprice', 'changeprice', 'cnt', 'creditcardprice', 'discount_price', 'discountprice', 'emoneyprice', 'etc', 'itemsubtotal', 'menu', 'menuqty_cnt', 'menutype_cnt', 'nm', 'num', 'othersvc_price', 'price', 'service_price', 'sub', 'sub_total', 'subtotal_price']

vocab size after add: 57580 (+54 field tokens)
decoder_start_token_id: 57525


In [11]:
# Sanity check: convert one sample to its target sequence and confirm it fits.
first_spec, first_train, _ = loaded_specs[0]
demo_parse = safe_parse(first_spec, first_train[0][first_spec["gt_col"]])
demo = json2token(demo_parse)
print("target sequence (first 400 chars):")
print(demo[:400])
ids = processor.tokenizer(demo, add_special_tokens=False).input_ids
print("\nlen(target tokens):", len(ids))
assert len(ids) < Cfg.MAX_LENGTH, "target longer than MAX_LENGTH — raise the cap"

target sequence (first 400 chars):
<s_menu><s_cnt>1 x</s_cnt><s_nm>Nasi Campur Bali</s_nm><s_price>75,000</s_price><sep/><s_cnt>1 x</s_cnt><s_nm>Bbk Bengil Nasi</s_nm><s_price>125,000</s_price><sep/><s_cnt>1 x</s_cnt><s_nm>MilkShake Starwb</s_nm><s_price>37,000</s_price><sep/><s_cnt>1 x</s_cnt><s_nm>Ice Lemon Tea</s_nm><s_price>24,000</s_price><sep/><s_cnt>1 x</s_cnt><s_nm>Nasi Ayam Dewata</s_nm><s_price>70,000</s_price><sep/><s_cn

len(target tokens): 373


## 5. PyTorch dataset

`MultiDonutDataset` flattens every loaded dataset split into one indexable view, pre-computes the target sequences (so we don't re-parse JSON each step), and applies light brightness/contrast/rotation augmentation when training.

In [12]:
def light_augment(image: Image.Image) -> Image.Image:
    """Mild perturbations that keep the receipt readable."""
    if random.random() < 0.5:
        image = ImageEnhance.Brightness(image).enhance(0.85 + random.random() * 0.3)
    if random.random() < 0.5:
        image = ImageEnhance.Contrast(image).enhance(0.85 + random.random() * 0.3)
    if random.random() < 0.3:
        angle = random.uniform(-3.0, 3.0)
        image = image.rotate(
            angle, resample=Image.BICUBIC, fillcolor=(255, 255, 255)
        )
    return image


class MultiDonutDataset(Dataset):
    def __init__(self, specs_with_splits, processor, max_length, task_token, augment=False):
        self.processor = processor
        self.max_length = max_length
        self.task_token = task_token
        self.augment = augment
        self.eos = processor.tokenizer.eos_token

        # (spec, split, local_idx) tuples for every kept example.
        self.entries = []
        self.targets = []
        for spec, split in specs_with_splits:
            gt_values = split[spec["gt_col"]]
            for i, gt_value in enumerate(gt_values):
                parsed = safe_parse(spec, gt_value)
                if parsed is None:
                    continue
                self.entries.append((spec, split, i))
                self.targets.append(task_token + json2token(parsed) + self.eos)

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        spec, split, local_idx = self.entries[idx]
        image = split[local_idx][spec["image_col"]]
        if image.mode != "RGB":
            image = image.convert("RGB")
        if self.augment:
            image = light_augment(image)

        pixel_values = self.processor(
            image, return_tensors="pt"
        ).pixel_values.squeeze(0)

        target = self.targets[idx]
        labels = self.processor.tokenizer(
            target,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            add_special_tokens=False,
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

In [13]:
train_specs = [(spec, train) for spec, train, _ in loaded_specs]
val_specs = [(spec, val) for spec, _, val in loaded_specs if val is not None]

train_ds = MultiDonutDataset(
    train_specs, processor, Cfg.MAX_LENGTH, Cfg.TASK_TOKEN, augment=Cfg.USE_AUGMENT
)
val_ds = MultiDonutDataset(
    val_specs, processor, Cfg.MAX_LENGTH, Cfg.TASK_TOKEN, augment=False
)
print("train:", len(train_ds), "val:", len(val_ds))

_item = train_ds[0]
print("pixel_values shape:", tuple(_item["pixel_values"].shape))
print("labels shape:", tuple(_item["labels"].shape))
print("first non-ignored target tokens:")
_labels = _item["labels"].clone()
_labels[_labels == -100] = processor.tokenizer.pad_token_id
print(processor.tokenizer.decode(_labels[:50]))

train: 800 val: 100
pixel_values shape: (3, 1280, 960)
labels shape: (768,)
first non-ignored target tokens:
<s_cord-v2> <s_menu> <s_cnt> 1 x </s_cnt> <s_nm> Nasi Campur Bali </s_nm> <s_price> 75,000 </s_price> <sep/> <s_cnt> 1 x </s_cnt> <s_nm> Bbk Bengil Nasi </s_nm> <s_price> 125,000 </s_price> <sep/> <s_cnt> 1 x </s_cnt> <s_nm> MilkShake Starwb </s_nm> <s_price> 37,000


## 6. Training

- Effective batch = 1 × 4 grad-accum = 4.
- bf16 on Ampere+, fp16 otherwise.
- Gradient checkpointing on (cuts encoder memory ~40%).

In [14]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

args = Seq2SeqTrainingArguments(
    output_dir=Cfg.OUTPUT_DIR,
    num_train_epochs=Cfg.EPOCHS,
    learning_rate=Cfg.LR,
    per_device_train_batch_size=Cfg.PER_DEVICE_BATCH,
    per_device_eval_batch_size=Cfg.PER_DEVICE_BATCH,
    gradient_accumulation_steps=Cfg.GRAD_ACC,
    weight_decay=Cfg.WEIGHT_DECAY,
    warmup_ratio=Cfg.WARMUP_RATIO,
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch" if len(val_ds) > 0 else "no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=len(val_ds) > 0,
    metric_for_best_model="eval_loss" if len(val_ds) > 0 else None,
    greater_is_better=False,
    logging_steps=20,
    predict_with_generate=False,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to=[],
    seed=Cfg.SEED,
)
print("bf16:", use_bf16, " fp16:", use_fp16)

bf16: True  fp16: False


In [15]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds if len(val_ds) > 0 else None,
)

train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
1,0.664700,0.401976
2,0.255500,0.238228
3,0.197500,0.171368
4,0.112400,0.173951
5,0.123700,0.159892


There were missing keys in the checkpoint model loaded: ['decoder.lm_head.weight'].


***** train metrics *****
  epoch                    =           5.0
  total_flos               = 11648653637GF
  train_loss               =        0.9135
  train_runtime            =    0:13:06.99
  train_samples_per_second =         5.083
  train_steps_per_second   =         1.271


## 7. Save final model + processor

In [16]:
final_dir = os.path.join(Cfg.OUTPUT_DIR, "final")
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
print("saved to", final_dir)

saved to ./donut-cord-finetuned/final


## 8. Quick inference demo

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

# Pull a sample from the first dataset's validation split.
_demo_spec, _, _demo_val = loaded_specs[0]
assert _demo_val is not None, "no validation split on the primary dataset"
test_sample = _demo_val[0]
image = test_sample[_demo_spec["image_col"]].convert("RGB")
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

decoder_input_ids = processor.tokenizer(
    Cfg.TASK_TOKEN, add_special_tokens=False, return_tensors="pt"
).input_ids.to(device)

with torch.inference_mode():
    out = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=Cfg.MAX_LENGTH,
        early_stopping=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=1,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )

seq = processor.batch_decode(out.sequences)[0]
seq = seq.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
seq = re.sub(r"<.*?>", "", seq, count=1).strip()  # strip the leading task token
parsed = processor.token2json(seq)

print("--- prediction ---")
print(json.dumps(parsed, indent=2, ensure_ascii=False)[:1500])
print("\n--- ground truth ---")
gt_demo = _demo_spec["gt_parser"](test_sample[_demo_spec["gt_col"]])
print(json.dumps(gt_demo, indent=2, ensure_ascii=False)[:1500])

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


--- prediction ---
{
  "menu": [
    {
      "cnt": "1",
      "nm": "REAL GANACHE",
      "price": "16,500"
    },
    {
      "cnt": "1",
      "nm": "EGG TART",
      "price": "13,000"
    },
    {
      "cnt": "1",
      "nm": "PIZZA TOAST",
      "price": "16,000"
    }
  ],
  "total": {
    "cashprice": "50,000",
    "changeprice": "4,500",
    "total_price": "45,500"
  }
}

--- ground truth ---
{
  "menu": [
    {
      "nm": "REAL GANACHE",
      "cnt": "1",
      "price": "16,500"
    },
    {
      "nm": "EGG TART",
      "cnt": "1",
      "price": "13,000"
    },
    {
      "nm": "PIZZA TOAST",
      "cnt": "1",
      "price": "16,000"
    }
  ],
  "total": {
    "total_price": "45,500",
    "cashprice": "50,000",
    "changeprice": "4,500"
  }
}
